# Multi-asset stat arb with stock-only universe, normalization, and PCA/KMeans selection

Pipeline:
1. Pull common-stock returns from WRDS CRSP.
2. Normalize the return panel for cross-sectional modeling.
3. Build stock embeddings with PCA.
4. Select a multi-stock basket with either PCA proximity or KMeans clustering.
5. Trade the residual basket as a market-neutral stat-arb spread.

In [ ]:
#Packages
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import wrds
import matplotlib.pyplot as plt
from dotenv import load_dotenv
import os

In [4]:
# Data Pipeline
# Create the WRDS client without auto-connecting so credentials can be saved first.
def get_wrds_data(query):
    import os
    import wrds
    from dotenv import load_dotenv

    load_dotenv()
    db = wrds.Connection(username=os.getenv("WRDS_USERNAME"), password=os.getenv("WRDS_PASSWORD"))
    try:
        data = db.raw_sql(query, date_cols=["date"])
    finally:
        db.close()
    return data


def build_stock_universe_query(symbols, start_date="2020-01-01"):
    ticker_list = ", ".join(f"'{symbol.upper()}'" for symbol in symbols)
    return f"""
    SELECT
        a.date,
        b.ticker,
        a.permno,
        a.prc,
        a.ret
    FROM crsp.dsf AS a
    JOIN crsp.stocknames AS b
      ON a.permno = b.permno
     AND a.date BETWEEN b.namedt AND b.nameenddt
    WHERE a.date >= '{start_date}'
      AND b.ticker IN ({ticker_list})
      AND b.shrcd IN (10, 11)
    """

# wrds cloud is another option but we will run it locally for now

In [5]:
symbols = ['AAPL', 'MSFT', 'GOOG', 'AMZN', 'META', 'NVDA', 'JPM', 'XOM']
price_query = build_stock_universe_query(symbols, start_date='2020-01-01')

df_prices = get_wrds_data(price_query)
df_prices = df_prices.sort_values(['ticker', 'date']).drop_duplicates(['date', 'ticker'])

available_symbols = sorted(df_prices['ticker'].dropna().unique())
print(f'Requested {len(symbols)} tickers, received {len(available_symbols)} common-stock series')
print('Available symbols:', available_symbols)

plt.figure(figsize=(12, 6))
for symbol in available_symbols:
    s = df_prices[df_prices['ticker'] == symbol].copy()
    s['cum_return'] = (1 + s['ret'].fillna(0.0)).cumprod() - 1
    plt.plot(s['date'], s['cum_return'], label=symbol)

plt.title('Cumulative Returns since 2020-01-01')
plt.xlabel('Date')
plt.ylabel('Cumulative Return of $1')
plt.legend()
plt.tight_layout()
plt.show()

OperationalError: (psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: fe_sendauth: no password supplied

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
# Prepare the price panel before any analysis uses it.
price_panel = df_prices.copy()
price_panel['ret'] = price_panel['ret'].fillna(0.0)
price_panel = price_panel.sort_values(['date', 'ticker']).drop_duplicates(['date', 'ticker'])

# Build a wide return panel for consistent downstream use.
returns_wide = price_panel.pivot(index='date', columns='ticker', values='ret').sort_index()
min_obs = max(60, int(0.8 * len(returns_wide)))
returns_wide = returns_wide.dropna(axis=1, thresh=min_obs).astype(float).fillna(0.0)

# Log returns from simple returns. Clip at -100% to avoid log1p domain errors.
log_returns_wide = np.log1p(returns_wide.clip(lower=-0.999999))
cum_log_returns_wide = log_returns_wide.cumsum()

# Standardize each ticker's return history across time for PCA and clustering.
standardized_returns_wide = (
    returns_wide.sub(returns_wide.mean(axis=0), axis=1)
    .div(returns_wide.std(axis=0).replace(0, np.nan), axis=1)
    .fillna(0.0)
)

standardized_log_returns_wide = (
    log_returns_wide.sub(log_returns_wide.mean(axis=0), axis=1)
    .div(log_returns_wide.std(axis=0).replace(0, np.nan), axis=1)
    .fillna(0.0)
)

# Each stock is an observation and each date is a feature for cross-sectional modeling.
stock_matrix = standardized_log_returns_wide.T
stock_matrix = stock_matrix.sub(stock_matrix.mean(axis=1), axis=0)
stock_matrix = stock_matrix.div(stock_matrix.std(axis=1).replace(0, np.nan), axis=0).fillna(0.0)

print('Prepared panels:')
print(f'  returns_wide shape: {returns_wide.shape}')
print(f'  log_returns_wide shape: {log_returns_wide.shape}')
print(f'  standardized_returns_wide shape: {standardized_returns_wide.shape}')
print(f'  standardized_log_returns_wide shape: {standardized_log_returns_wide.shape}')
print(f'  stock_matrix shape: {stock_matrix.shape}')

In [ ]:
# Correlation and PCA quality checks for the normalized stock universe.
normalized_returns = standardized_log_returns_wide.copy()
corr_matrix = normalized_returns.corr()

n_components = min(5, stock_matrix.shape[0], stock_matrix.shape[1])
pca = PCA(n_components=n_components, random_state=0)
stock_embeddings = pca.fit_transform(stock_matrix)

embedding_frame = pd.DataFrame(stock_embeddings[:, :2], index=stock_matrix.index, columns=['pc1', 'pc2'])
explained_variance = pd.Series(
    pca.explained_variance_ratio_,
    index=[f'pc{i + 1}' for i in range(n_components)],
    name='explained_variance'
)

print('Explained variance by component:')
print(explained_variance.round(4))
print('\nTicker correlation matrix:')
print(corr_matrix.round(3))

NameError: name 'df_prices' is not defined

In [ ]:
selection_method = 'kmeans'  # or 'pca'
basket_size = 5

if selection_method == 'kmeans':
    n_clusters = min(4, len(stock_matrix.index))
    kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init='auto')
    cluster_labels = kmeans.fit_predict(stock_embeddings)
    cluster_mapping = pd.Series(cluster_labels, index=stock_matrix.index, name='cluster')
    selected_cluster = cluster_mapping.value_counts().idxmax()
    basket_tickers = cluster_mapping[cluster_mapping == selected_cluster].index.tolist()
else:
    center = embedding_frame.median()
    distances = ((embedding_frame[['pc1', 'pc2']] - center) ** 2).sum(axis=1).sort_values()
    basket_tickers = distances.head(max(basket_size, 2)).index.tolist()
    cluster_mapping = pd.Series(index=stock_matrix.index, dtype='float64', name='cluster')
    selected_cluster = 'pca_central_basket'

basket_tickers = list(dict.fromkeys(basket_tickers))

print(f'Selection method: {selection_method}')
print(f'Selected basket: {basket_tickers}')
print(f'Selected group: {selected_cluster}')

In [ ]:
from statsmodels.tsa.stattools import adfuller

if len(basket_tickers) < 2:
    raise ValueError('Need at least two stocks in the basket.')

basket_returns = standardized_log_returns_wide[basket_tickers].fillna(0.0)
basket_pca = PCA(n_components=1, random_state=0)
basket_pca.fit(basket_returns)

raw_weights = pd.Series(basket_pca.components_[0], index=basket_tickers, name='raw_weight')
basket_weights = raw_weights - raw_weights.mean()
if np.isclose(basket_weights.abs().sum(), 0.0):
    basket_weights = pd.Series(1.0, index=basket_tickers)
basket_weights = basket_weights / basket_weights.abs().sum()
basket_weights.name = 'weight'

basket_spread = (cum_log_returns_wide[basket_tickers].fillna(0.0) @ basket_weights).rename('basket_spread')
basket_spread_mean = basket_spread.rolling(20, min_periods=5).mean()
basket_spread_std = basket_spread.rolling(20, min_periods=5).std()
basket_spread_zscore = ((basket_spread - basket_spread_mean) / basket_spread_std).replace([np.inf, -np.inf], np.nan)

if basket_spread.dropna().nunique() >= 3:
    adf_stat, adf_pvalue, *_ = adfuller(basket_spread.dropna(), autolag='AIC')
else:
    adf_stat, adf_pvalue = np.nan, np.nan

basket_summary = pd.DataFrame({
    'basket_spread': basket_spread,
    'basket_spread_zscore': basket_spread_zscore,
})

print('Basket weights:')
print(basket_weights.round(4))
print(f'ADF stat on basket spread: {adf_stat:.4f}')
print(f'ADF p-value on basket spread: {adf_pvalue:.4f}')
print(basket_summary.tail())

In [ ]:
entry_z = 2.0
exit_z = 0.5

position = pd.Series(0.0, index=basket_spread.index, name='position')
for idx in range(1, len(position)):
    prev_position = position.iloc[idx - 1]
    zscore = basket_spread_zscore.iloc[idx]

    if np.isnan(zscore):
        position.iloc[idx] = prev_position
    elif prev_position == 0.0:
        if zscore > entry_z:
            position.iloc[idx] = -1.0
        elif zscore < -entry_z:
            position.iloc[idx] = 1.0
        else:
            position.iloc[idx] = 0.0
    elif prev_position > 0.0 and zscore >= -exit_z:
        position.iloc[idx] = prev_position
    elif prev_position < 0.0 and zscore <= exit_z:
        position.iloc[idx] = prev_position
    else:
        position.iloc[idx] = 0.0

basket_strategy_return = position.shift(1).fillna(0.0) * basket_spread.diff().fillna(0.0)
basket_equity_curve = (1 + basket_strategy_return).cumprod()

signal_changes = position.diff().fillna(position).ne(0)
order_rows = []
for trade_id, trade_date in enumerate(position.index[signal_changes], start=1):
    trade_direction = position.loc[trade_date]
    action = 'enter_long' if trade_direction > 0 else 'enter_short' if trade_direction < 0 else 'flat'
    for ticker, weight in basket_weights.items():
        order_rows.append({
            'timestamp': trade_date,
            'strategy': f'{selection_method}_multi_asset_stat_arb',
            'cluster_id': selected_cluster,
            'base_ticker': ticker,
            'hedge_ticker': pd.NA,
            'signal_id': f'{trade_date:%Y%m%d}_{trade_id}',
            'action': action,
            'side': 'buy' if trade_direction * weight > 0 else 'sell',
            'quantity': float(abs(weight) * 100.0),
            'price': np.nan,
            'notional': float(abs(weight)),
            'alpha': np.nan,
            'beta': float(weight),
            'spread': float(basket_spread.loc[trade_date]),
            'spread_zscore': float(basket_spread_zscore.loc[trade_date]) if pd.notna(basket_spread_zscore.loc[trade_date]) else np.nan,
            'order_id': f'{trade_date:%Y%m%d}_{ticker}_{trade_id}',
        })

basket_order_log = pd.DataFrame(order_rows)

print(f'Strategy cumulative return: {basket_equity_curve.iloc[-1] - 1:.4f}')
print(f'Active trading days: {(position != 0).sum()}')
print(basket_order_log.head())

In [ ]:
from pathlib import Path

# Persist every generated order row so validation can replay the exact execution later.
order_log_columns = [
    'timestamp',
    'strategy',
    'cluster_id',
    'base_ticker',
    'hedge_ticker',
    'signal_id',
    'action',
    'side',
    'quantity',
    'price',
    'notional',
    'alpha',
    'beta',
    'spread',
    'spread_zscore',
    'order_id'
]

if 'basket_order_log' in globals() and isinstance(basket_order_log, pd.DataFrame):
    order_log_df = basket_order_log.copy()
elif 'order_log' in globals() and isinstance(order_log, pd.DataFrame):
    order_log_df = order_log.copy()
elif 'order_rows' in globals():
    order_log_df = pd.DataFrame(order_rows)
else:
    order_log_df = pd.DataFrame(columns=order_log_columns)

for column in order_log_columns:
    if column not in order_log_df.columns:
        order_log_df[column] = pd.NA

order_log_df = order_log_df[order_log_columns]

order_log_path = Path('order_log.csv')
order_log_df.to_csv(order_log_path, index=False)

print(f'Wrote {len(order_log_df)} order rows to {order_log_path.resolve()}')
print(order_log_df.head())

# Need to do the OU process into stopping problem logic

Solving entry times everything, + maybe making a streaming variant so i can test it on alpaca, change wrds access and introduce execution layer